# Week 8-C: Counterfactual Passage Utility Modeling
## Predicting Marginal Promotion Utility $(q, y_0, D_i) \to \Delta_i$

Trains and evaluates:
- **U0**: Tabular Baseline (GBDT + Logistic) on engineered features.
- **U1**: Pre-generation Query/Passage Cross-Encoder.
- **U2**: Draft-Conditioned Cross-Encoder $[q, y_0, D_i]$ with 4 multitask heads.
- **U3**: Listwise Draft-Aware Utility Router (Mistral-7B QLoRA).


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from src.week8.utility_dataset import build_utility_dataset
from src.week8.utility_models import TabularUtilityModel_U0

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
print("W8-C environment ready.")


In [ ]:
# Build multi-task utility dataset
prom_path = REPO_ROOT / "results/week8/surfaces/promotion_surface_internal_val.jsonl"
draft_path = REPO_ROOT / "results/week8/drafts/draft_diagnostics_internal_val.jsonl"

df_prom = pd.read_json(prom_path, lines=True)
df_draft = pd.read_json(draft_path, lines=True)
df_util = build_utility_dataset(df_prom, df_draft)
print(f"Dataset assembled with {len(df_util)} rows.")


In [ ]:
# Train U0 Tabular Baseline
feat_cols = ["base_context_tokens", "promoted_context_tokens", "semantic_rank"]
X = df_util[feat_cols].values
y_help = df_util["target_help_binary"].values
y_hurt = df_util["target_hurt_binary"].values
y_delta = df_util["target_continuous_delta"].values
y_3way = df_util["target_3way_label"].values

u0 = TabularUtilityModel_U0()
u0.fit(X, y_help, y_hurt, y_delta, y_3way)
preds_u0 = u0.predict(X)
print("U0 Training Complete.")
